# MLP VQ-VAE training

In [ ]:
#Import custom classes and libraries
from src.Data.data_loading import ParquetDataModule
from src.Data.data_loading import ParquetFeatureDataset
from src.Models.VQ_VAE_MLP import VQVAE_MLP 

import yaml
import numpy as np
import fastjet
from tqdm import tqdm
import torch
import lightning as L
import matplotlib.pyplot as plt


#Open config file
with open("Configs/config.yaml") as f:
    config = yaml.safe_load(f)

#Extract config info
dirs_train = config["data"]["train_path"]
dirs_val = config["data"]["val_path"]
dirs_test = config["data"]["test_path"]

features_cols = config["data"]["features"]

input_dim = config["model"]["input_dim"]
hidden_dim = config["model"]["hidden_dim"]
latent_dim = config["model"]["latent_dim"]
codebook_size = config["model"]["codebook_size"]

lr = config["training"]["lr"]
max_epochs = config["training"]["max_epochs"]


#Initialization of itarable dataset (train dataset) containing the constituents' features (eta, phi, pT)
#In particular I am currently using the first 2 parquet files
dataset = ParquetFeatureDataset(
    parquet_dirs=dirs_train,
    features=features_cols,
    preprocessing=True
    #max_particles=256,
    #batch_size=32
)

#Initialization of the lightining datamodule
data_module = ParquetDataModule(
    parquet_dirs_train=dirs_train, 
    parquet_dirs_val=dirs_val,
    parquet_dirs_test=dirs_test,
    features=features_cols
    #window_particles=256,
    #num_workers=0
)


In [3]:
#Model
model = VQVAE_MLP(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    latent_dim=latent_dim,
    codebook_size=codebook_size,
    lr=lr
)


#Trainer (PyTorch Lightning)
trainer = L.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",   # CPU/GPU automatico
    devices="auto",
    log_every_n_steps=10
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [4]:
#Train
trainer.fit(model, datamodule=data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


2026-04-20 16:16:34.050905: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-20 16:16:34.139114: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-20 16:16:34.164256: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-20 16:16:34.336583: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-20 16:16:35.808492: W tensorflow/compiler/tf2

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder │ Encoder        │ 33.5 K │ train │     0 │
│ 1 │ decoder │ Decoder        │ 33.3 K │ train │     0 │
│ 2 │ vq      │ VectorQuantize │      0 │ train │     0 │
└───┴─────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 66.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 66.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/francesco/anaconda3/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: 
The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/home/francesco/anaconda3/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: 
The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=50` reached.


In [5]:
#Validation
trainer.validate(model, datamodule=data_module)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/francesco/anaconda3/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      val_commit_loss      │            0.0            │
│         val_loss          │    0.11931091547012329    │
│      val_recon_loss       │    0.11931091547012329    │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 0.11931091547012329,
  'val_recon_loss': 0.11931091547012329,
  'val_commit_loss': 0.0}]

In [6]:
#Test 
trainer.test(model, datamodule=data_module)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/francesco/anaconda3/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     test_commit_loss      │            0.0            │
│         test_loss         │    0.11929228901863098    │
│      test_recon_loss      │    0.11929228901863098    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.11929228901863098,
  'test_recon_loss': 0.11929228901863098,
  'test_commit_loss': 0.0}]

## Trained model

In [9]:
model = VQVAE_MLP.load_from_checkpoint("lightning_logs/version_16/checkpoints/epoch=49-step=27150.ckpt")
 
trainer = L.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",   # CPU/GPU automatico
    devices="auto",
    log_every_n_steps=10
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [10]:
trainer.validate(model, datamodule=data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/francesco/anaconda3/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      val_commit_loss      │            0.0            │
│         val_loss          │    0.11931091547012329    │
│      val_recon_loss       │    0.11931091547012329    │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 0.11931091547012329,
  'val_recon_loss': 0.11931091547012329,
  'val_commit_loss': 0.0}]